In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

In [ ]:
#########################
# 1. Load the Fine-Tuned Model
#########################
# IMPORTANT: Make sure you pass the local path you used to save your model
# e.g. new_model_local = "Gemma-3-12B-it-FirstResponder"
FINETUNED_MODEL_PATH = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit"
FINETUNED_MODEL_PATH = "./Gemma_finetuned 2000steos Arapro"

torch.cuda.empty_cache()

print("Loading the fine-tuned model...")
model, tokenizer = FastModel.from_pretrained(
    model_name = FINETUNED_MODEL_PATH,
    max_seq_length = 20000,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    # token = "hf_..."  # If your model is gated and requires a token
)

# Because we used the gemma-3 chat template, fetch it again
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",  # same template used during finetuning
)

model.eval()

In [ ]:
#########################
# 2. Load the Test Data
#########################
TEST_FILE = "./progressive_test_20.csv"

test_df = pd.read_csv(TEST_FILE)

# We expect columns: "input" and "output" (help-seeker input, first-responder reference)
test_df = test_df.dropna(subset=["input", "output"])  # just in case
test_df = test_df.sample(n=3000, random_state=42)

In [ ]:
#########################
# 3. Define System Prompt
#########################
system_prompt = (
    "أنت مساعد دعم نفسي متعاطف ومليء بالرحمة، يقدم دعمًا عاطفيًا من خلال محادثات نصية للأشخاص الذين يطلبون المساعدة. "
    "دورك هو الاستماع بإنصات، وتأكيد مشاعرهم، وتقديم الدعم العاطفي. "
    "شجعهم بلطف، واطرح أسئلة مفتوحة، ووجّه المستخدمين نحو استراتيجيات تأقلم إيجابية. "
    "تجنب تقديم تشخيصات طبية أو توصيات لعلاج طبي. "
    "إذا ذكر المستخدم أنه في ضائقة فورية أو أشار إلى إيذاء النفس، فاقترح بلطف التواصل مع مختص في الصحة النفسية أو الاتصال بخدمات الطوارئ. "
    "حافظ دائمًا على مساحة آمنة وغير حُكمية يمكن للمستخدمين المشاركة فيها بحرية."
)


def build_prompt(user_input: str):

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user",   "content": [{"type": "text", "text": user_input}]}
    ]
    
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True)


    


In [ ]:
import json
#########################
# 4. Generate Predictions and Save
#########################
output_data = []

print("Running inference on test set...")
curr_idx = 0
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if curr_idx % 2 == 0:
        with open(f'./Gemma_finetuned 2000steos Arapro/Arapro_test_output.json', 'w') as json_file:
            json.dump([{"steps": curr_idx}]+output_data, json_file, indent=4)
    curr_idx += 1
    user_text = row["input"]
    reference_text = row["output"]
    
    prompt_text = build_prompt(user_text)

    inputs = tokenizer(text=prompt_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            do_sample=True
        )
    
    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    output_data.append({
        "input": user_text,
        "reference": reference_text,
        "prediction": gen_text
    })


In [ ]:
# generate the same json file but for each prediction split by \nmodel\n
output_data_split = []
for data in output_data:
    user_text = data["input"]
    reference_text = data["reference"]
    prediction_text = data["prediction"].split("\nmodel\n")[-1]
    
    output_data_split.append({
        "input": user_text,
        "reference": reference_text,
        "prediction": prediction_text
    })
# Save to JSON
with open("./Gemma_finetuned 2000steos Arapro.json", "w", encoding="utf-8") as f:
    json.dump(output_data_split, f, ensure_ascii=False, indent=2)

Saved outputs to generation_outputs.json ✅
